In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/karineoliviapermana/atis-train-w/atis.train.w-intent.iob
/kaggle/input/datasets/karineoliviapermana/atis-train/atis.train.pkl


In [3]:
# Ubah impor Keras lama ini:
# from keras.preprocessing.text import Tokenizer
# from keras.preprocessing.sequence import pad_sequences
# from keras.utils import to_categorical
# from keras.layers import Dense, Input, GlobalMaxPooling1D
# from keras.layers import Conv1D, MaxPooling1D, Embedding, LSTM
# from keras.models import Model, Sequential
# from keras.initializers import Constant

# Menjadi impor TensorFlow Keras baru:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Dense, Input, GlobalMaxPooling1D
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Embedding, LSTM
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.initializers import Constant

In [4]:
pip install tf-keras

Note: you may need to restart the kernel to use updated packages.


In [5]:
from Data.utils import fetch_data, read_method

sents,labels,intents = fetch_data('Data/data2/atis.train.w-intent.iob')

train_sentences = [" ".join(i) for i in sents]

train_texts = train_sentences
train_labels= intents.tolist()

vals = []

for i in range(len(train_labels)):
    if "#" in train_labels[i]:
        vals.append(i)
        
for i in vals[::-1]:
    train_labels.pop(i)
    train_texts.pop(i)

print ("Number of training sentences :",len(train_texts))
print ("Number of unique intents :",len(set(train_labels)))

for i in zip(train_texts[:5], train_labels[:5]):
    print(i)

Number of training sentences : 4952
Number of unique intents : 17
('i want to fly from boston at 838 am and arrive in denver at 1110 in the morning', 'atis_flight')
('what flights are available from pittsburgh to baltimore on thursday morning', 'atis_flight')
('what is the arrival time in san francisco for the 755 am flight leaving washington', 'atis_flight_time')
('cheapest airfare from tacoma to orlando', 'atis_airfare')
('round trip fares from pittsburgh to philadelphia under 1000 dollars', 'atis_airfare')


In [ ]:
from Data.utils import fetch_data, read_method

sents,labels,intents = fetch_data('Data/data2/atis.test.w-intent.iob')

test_sentences = [" ".join(i) for i in sents]

test_texts = test_sentences
test_labels = intents.tolist()

new_labels = set(test_labels) - set(train_labels)

vals = []

for i in range(len(test_labels)):
    if "#" in test_labels[i]:
        vals.append(i)
    elif test_labels[i] in new_labels:
        print(test_labels[i])
        vals.append(i)
        
for i in vals[::-1]:
    test_labels.pop(i)
    test_texts.pop(i)

print ("Number of testing sentences :",len(test_texts))
print ("Number of unique intents :",len(set(test_labels)))

for i in zip(test_texts[:5], test_labels[:5]):
    print(i)

In [6]:
MAX_SEQUENCE_LENGTH = 300
MAX_NUM_WORDS = 20000 
EMBEDDING_DIM = 100 
VALIDATION_SPLIT = 0.3

In [8]:
tokenizer = Tokenizer(num_words=MAX_NUM_WORDS)
tokenizer.fit_on_texts(train_texts)
train_sequences = tokenizer.texts_to_sequences(train_texts) #Converting text to a vector of word indexes
test_sequences = tokenizer.texts_to_sequences(test_texts)
word_index = tokenizer.word_index
print('Found %s unique tokens.' % len(word_index))

Found 897 unique tokens.


In [11]:
import numpy as np

# Pastikan train_labels dan test_labels berupa list 1D string, 
# contoh jika bentuknya array bersarang, kita bisa ubah atau pastikan konversinya:
# (Sesuaikan dengan variabel aslimu, pastikan isinya string murni per baris)

# Jika train_labels berbentuk numpy array dari objek/list bersarang:
train_labels = np.array([str(i) for i in train_labels])
test_labels = np.array([str(i) for i in test_labels])

le = LabelEncoder()
le.fit(train_labels)
train_labels = le.transform(train_labels)
test_labels = le.transform(test_labels)

In [12]:
#Converting this to sequences to be fed into neural network. Max seq. len is 1000 as set earlier
 #initial padding of 0s, until vector is of size MAX_SEQUENCE_LENGTH
trainvalid_data = pad_sequences(train_sequences, maxlen=MAX_SEQUENCE_LENGTH)
test_data = pad_sequences(test_sequences, maxlen=MAX_SEQUENCE_LENGTH)
trainvalid_labels = to_categorical(train_labels)

test_labels = to_categorical(np.asarray(test_labels), num_classes= trainvalid_labels.shape[1])

# split the training data into a training set and a validation set
indices = np.arange(trainvalid_data.shape[0])
np.random.shuffle(indices)
trainvalid_data = trainvalid_data[indices]
trainvalid_labels = trainvalid_labels[indices]
num_validation_samples = int(VALIDATION_SPLIT * trainvalid_data.shape[0])
x_train = trainvalid_data[:-num_validation_samples]
y_train = trainvalid_labels[:-num_validation_samples]
x_val = trainvalid_data[-num_validation_samples:]
y_val = trainvalid_labels[-num_validation_samples:]
#This is the data we will use for CNN and RNN training
print('Splitting the train data into train and valid is done')

Splitting the train data into train and valid is done


In [14]:
print('Preparing embedding matrix.')

# first, build index mapping words in the embeddings set
# to their embedding vector

# Download GloVe 6B from here: https://nlp.stanford.edu/projects/glove/
BASE_DIR = 'Data'
GLOVE_DIR = os.path.join(BASE_DIR, 'glove.6B')

embeddings_index = {}
with open(os.path.join(GLOVE_DIR, 'glove.6B.100d.txt'), encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

print('Found %s word vectors in Glove embeddings.' % len(embeddings_index))
#print(embeddings_index["google"])

# prepare embedding matrix - rows are the words from word_index, columns are the embeddings of that word from glove.
num_words = min(MAX_NUM_WORDS, len(word_index)) + 1
embedding_matrix = np.zeros((num_words, EMBEDDING_DIM))
for word, i in word_index.items():
    if i > MAX_NUM_WORDS:
        continue
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        # words not found in embedding index will be all-zeros.
        embedding_matrix[i] = embedding_vector

# load these pre-trained word embeddings into an Embedding layer
# note that we set trainable = False so as to keep the embeddings fixed
embedding_layer = Embedding(num_words,
                            EMBEDDING_DIM,
                            embeddings_initializer=Constant(embedding_matrix),
                            trainable=False)
print("Preparing of embedding matrix is done")

Preparing embedding matrix.
Found 400000 word vectors in Glove embeddings.
Preparing of embedding matrix is done


In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, Conv1D, MaxPooling1D, GlobalMaxPooling1D, Dense
from tensorflow.keras.initializers import Constant

# 1. Definisikan model CNN menggunakan Input() dan pastikan GlobalMaxPooling1D ada
cnnmodel = Sequential()
cnnmodel.add(Input(shape=(MAX_SEQUENCE_LENGTH,)))
cnnmodel.add(Embedding(
    num_words, 
    EMBEDDING_DIM, 
    embeddings_initializer=Constant(embedding_matrix), 
    trainable=False
))

cnnmodel.add(Conv1D(128, 5, activation='relu'))
cnnmodel.add(MaxPooling1D(5))
cnnmodel.add(Conv1D(128, 5, activation='relu'))
cnnmodel.add(MaxPooling1D(5))
cnnmodel.add(Conv1D(128, 5, activation='relu'))

# Layer ini wajib ada untuk mengubah output 3D conv1d menjadi 2D sebelum masuk ke Dense
cnnmodel.add(GlobalMaxPooling1D()) 

cnnmodel.add(Dense(128, activation='relu'))
cnnmodel.add(Dense(17, activation='softmax')) # Sesuaikan angka 17 dengan jumlah total kelas intent kamu

# 2. Compile model
cnnmodel.compile(
    loss='categorical_crossentropy',
    optimizer='rmsprop',
    metrics=['acc']
)

# 3. Tampilkan ringkasan arsitektur model
cnnmodel.summary()

# 4. Train model (ubah epochs menjadi 5 atau 10 agar hasilnya lebih optimal)
history = cnnmodel.fit(
    x_train, y_train,
    batch_size=128,
    epochs=5, 
    validation_data=(x_val, y_val)
)

# 5. Evaluasi pada test set
score, acc = cnnmodel.evaluate(test_data, test_labels)
print('Test accuracy with CNN:', acc)

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_10 (Embedding)        │ (None, 300, 100)       │        89,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_22 (Conv1D)              │ (None, 296, 128)       │        64,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_12 (MaxPooling1D) │ (None, 59, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_23 (Conv1D)              │ (None, 55, 128)        │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_13 (MaxPooling1D) │ (None, 11, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_24 (Conv1D)              │ (None, 7, 128)         │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_6          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 17)             │         2,193 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 336,729 (1.28 MB)

 Trainable params: 246,929 (964.57 KB)

 Non-trainable params: 89,800 (350.78 KB)

Epoch 1/5
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 99ms/step - acc: 0.7116 - loss: 1.1455 - val_acc: 0.7461 - val_loss: 0.8280
Epoch 2/5
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - acc: 0.7975 - loss: 0.7294 - val_acc: 0.8444 - val_loss: 0.6943
Epoch 3/5
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - acc: 0.8722 - loss: 0.4996 - val_acc: 0.8963 - val_loss: 0.4648
Epoch 4/5
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - acc: 0.9022 - loss: 0.3704 - val_acc: 0.9165 - val_loss: 0.3276
Epoch 5/5
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - acc: 0.9305 - loss: 0.2562 - val_acc: 0.9293 - val_loss: 0.3138
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - acc: 0.0470 - loss: 13.6874   
Test accuracy with CNN: 0.04703247547149658


In [26]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, Conv1D, MaxPooling1D, GlobalMaxPooling1D, Dense
from tensorflow.keras.initializers import Constant

print("Defining and training a CNN model using pre-trained GloVe embeddings")
cnnmodel = Sequential()

# 1. Gunakan Input() di awal agar shape model jelas dan tidak '?'
cnnmodel.add(Input(shape=(MAX_SEQUENCE_LENGTH,)))

# 2. Gunakan Embedding dengan matrix GloVe yang sudah kita siapkan
cnnmodel.add(Embedding(
    num_words,
    EMBEDDING_DIM,
    embeddings_initializer=Constant(embedding_matrix),
    trainable=False
))

# 3. Layer CNN & Pooling
cnnmodel.add(Conv1D(128, 5, activation='relu'))
cnnmodel.add(MaxPooling1D(5))
cnnmodel.add(Conv1D(128, 5, activation='relu'))
cnnmodel.add(MaxPooling1D(5))
cnnmodel.add(Conv1D(128, 5, activation='relu'))
cnnmodel.add(GlobalMaxPooling1D())

# 4. Dense Layers
cnnmodel.add(Dense(128, activation='relu'))
cnnmodel.add(Dense(len(trainvalid_labels[0]), activation='softmax'))

# 5. Compile model
cnnmodel.compile(
    loss='categorical_crossentropy',
    optimizer='rmsprop',
    metrics=['acc']
)

# Tampilkan ringkasan model (sekarang angkanya akan keluar lengkap, tidak '?' lagi)
cnnmodel.summary()

# 6. Train the model (ubah epochs menjadi 5 agar model sempat belajar)
history = cnnmodel.fit(
    x_train, y_train,
    batch_size=128,
    epochs=5, 
    validation_data=(x_val, y_val)
)

# 7. Evaluate on test set
score, acc = cnnmodel.evaluate(test_data, test_labels)
print('Test accuracy with CNN:', acc)

Defining and training a CNN model using pre-trained GloVe embeddings


Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_13 (Embedding)        │ (None, 300, 100)       │        89,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_31 (Conv1D)              │ (None, 296, 128)       │        64,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_18 (MaxPooling1D) │ (None, 59, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_32 (Conv1D)              │ (None, 55, 128)        │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_19 (MaxPooling1D) │ (None, 11, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_33 (Conv1D)              │ (None, 7, 128)         │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_9          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 17)             │         2,193 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 336,729 (1.28 MB)

 Trainable params: 246,929 (964.57 KB)

 Non-trainable params: 89,800 (350.78 KB)

Epoch 1/5
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 100ms/step - acc: 0.7113 - loss: 1.1057 - val_acc: 0.7791 - val_loss: 1.1197
Epoch 2/5
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - acc: 0.7918 - loss: 0.7525 - val_acc: 0.8586 - val_loss: 0.5693
Epoch 3/5
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - acc: 0.8711 - loss: 0.5016 - val_acc: 0.9030 - val_loss: 0.4461
Epoch 4/5
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - acc: 0.8944 - loss: 0.4011 - val_acc: 0.9185 - val_loss: 0.3428
Epoch 5/5
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - acc: 0.9328 - loss: 0.2447 - val_acc: 0.9212 - val_loss: 0.3450
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - acc: 0.0896 - loss: 12.1250   
Test accuracy with CNN: 0.08958566933870316


In [28]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

print("Defining and training an LSTM model, training embedding layer on the fly")

rnnmodel = Sequential()

# 1. Gunakan Input() di awal agar Sequential model tahu bentuk dimensi masukannya
rnnmodel.add(Input(shape=(MAX_SEQUENCE_LENGTH,)))

# 2. Layer Embedding dilatih dari awal (on-the-fly) dengan ukuran 128
rnnmodel.add(Embedding(MAX_NUM_WORDS, 128))

# 3. Layer LSTM & Dropout
rnnmodel.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))

# 4. Dense Output Layer menggunakan softmax & categorical_crossentropy untuk multi-kelas (ATIS intents)
rnnmodel.add(Dense(len(trainvalid_labels[0]), activation='softmax'))

# 5. Compile model
rnnmodel.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Tampilkan ringkasan model (hasilnya akan persis seperti di foto kedua)
rnnmodel.summary()

print('Training the RNN')
history = rnnmodel.fit(
    x_train, y_train,
    batch_size=32,
    epochs=5,  # Bisa disesuaikan jumlah epoch-nya
    validation_data=(x_val, y_val)
)

# Evaluasi pada test set
score, acc = rnnmodel.evaluate(test_data, test_labels, batch_size=32)
print('Test accuracy with RNN:', acc)

Defining and training an LSTM model, training embedding layer on the fly


Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_15 (Embedding)        │ (None, 300, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 17)             │         2,193 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,693,777 (10.28 MB)

 Trainable params: 2,693,777 (10.28 MB)

 Non-trainable params: 0 (0.00 B)

Training the RNN
Epoch 1/5
109/109 ━━━━━━━━━━━━━━━━━━━━ 107s 954ms/step - accuracy: 0.7346 - loss: 1.1767 - val_accuracy: 0.7912 - val_loss: 0.9195
Epoch 2/5
109/109 ━━━━━━━━━━━━━━━━━━━━ 103s 950ms/step - accuracy: 0.8137 - loss: 0.7131 - val_accuracy: 0.8808 - val_loss: 0.5156
Epoch 3/5
109/109 ━━━━━━━━━━━━━━━━━━━━ 103s 949ms/step - accuracy: 0.8815 - loss: 0.4489 - val_accuracy: 0.9017 - val_loss: 0.4054
Epoch 4/5
109/109 ━━━━━━━━━━━━━━━━━━━━ 104s 955ms/step - accuracy: 0.9143 - loss: 0.3099 - val_accuracy: 0.9327 - val_loss: 0.2799
Epoch 5/5
109/109 ━━━━━━━━━━━━━━━━━━━━ 103s 946ms/step - accuracy: 0.9397 - loss: 0.2102 - val_accuracy: 0.9434 - val_loss: 0.2358
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 167ms/step - accuracy: 0.0728 - loss: 10.0464
Test accuracy with RNN: 0.07278835028409958


In [29]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense

print("Defining and training an LSTM model using pre-trained embedding layer")

rnnmodel2 = Sequential()

# 1. Gunakan Input() di awal agar Sequential model tahu bentuk dimensi masukannya
rnnmodel2.add(Input(shape=(MAX_SEQUENCE_LENGTH,)))

# 2. Masukkan embedding_layer yang sudah didefinisikan sebelumnya di cell atas
rnnmodel2.add(embedding_layer)

# 3. Layer LSTM & Dropout
rnnmodel2.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))

# 4. Dense Output Layer (gunakan softmax & kategori multi-kelas)
rnnmodel2.add(Dense(len(trainvalid_labels[0]), activation='softmax'))

# 5. Compile model dengan categorical_crossentropy
rnnmodel2.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Tampilkan ringkasan model
rnnmodel2.summary()

print('Training the RNN')
history = rnnmodel2.fit(
    x_train, y_train,
    batch_size=32,
    epochs=5,  # Naikkan epoch agar modelnya sempat belajar dengan optimal
    validation_data=(x_val, y_val)
)

# Evaluasi pada test set
score, acc = rnnmodel2.evaluate(test_data, test_labels, batch_size=32)
print('Test accuracy with RNN:', acc)

Defining and training an LSTM model using pre-trained embedding layer


Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 300, 100)       │        89,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 128)            │       117,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 17)             │         2,193 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 209,241 (817.35 KB)

 Trainable params: 119,441 (466.57 KB)

 Non-trainable params: 89,800 (350.78 KB)

Training the RNN
Epoch 1/5
109/109 ━━━━━━━━━━━━━━━━━━━━ 98s 866ms/step - accuracy: 0.7681 - loss: 0.9556 - val_accuracy: 0.8768 - val_loss: 0.5112
Epoch 2/5
109/109 ━━━━━━━━━━━━━━━━━━━━ 93s 854ms/step - accuracy: 0.8667 - loss: 0.5024 - val_accuracy: 0.9064 - val_loss: 0.3563
Epoch 3/5
109/109 ━━━━━━━━━━━━━━━━━━━━ 94s 859ms/step - accuracy: 0.9086 - loss: 0.3351 - val_accuracy: 0.9401 - val_loss: 0.2659
Epoch 4/5
109/109 ━━━━━━━━━━━━━━━━━━━━ 93s 856ms/step - accuracy: 0.9270 - loss: 0.2616 - val_accuracy: 0.9481 - val_loss: 0.2172
Epoch 5/5
109/109 ━━━━━━━━━━━━━━━━━━━━ 94s 861ms/step - accuracy: 0.9446 - loss: 0.2025 - val_accuracy: 0.9542 - val_loss: 0.1937
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 172ms/step - accuracy: 0.0403 - loss: 10.4288
Test accuracy with RNN: 0.04031354933977127
